The focus of the project is on detecting phishing url links. 
The two datasets: 
1. primary dataset: https://data.mendeley.com/datasets/vfszbj9b36/1
2. secondary dataset: kaggle dataset, kaggle.com/datasets/sid321axn/malicious-urls-dataset 

In [1]:
import pandas as pd
import numpy as np

In [2]:
phish_df = pd.read_csv("../data/Phishing URLs.csv")
phish_df = phish_df.rename(columns={"Type": "type"})
phish_df["type"] = phish_df["type"].str.lower()
len(phish_df)

54807

In [3]:
real_df = pd.read_csv("../data/URL dataset.csv")
real_df.head()

,url,type
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate


In [4]:
kaggle_df = pd.read_csv("../data/kaggle_dataset.csv")
len(kaggle_df)

651191

## first modeling with the 


In [5]:
# combine the two dataset
prim_df = pd.concat([phish_df, real_df])
prim_df.head()

,url,type
0,https://docs.google.com/presentation/d/e/2PACX...,phishing
1,https://btttelecommunniccatiion.weeblysite.com/,phishing
2,https://kq0hgp.webwave.dev/,phishing
3,https://brittishtele1bt-69836.getresponsesite....,phishing
4,https://bt-internet-105056.weeblysite.com/,phishing


In [6]:
prim_df["type"].value_counts()

type
legitimate    345738
phishing      159245
Name: count, dtype: int64

In [7]:
# want to recode legitimate as benign
prim_df.loc[prim_df["type"] == "legitimate", "type"] = "benign"

In [8]:
prim_df.columns

Index(['url', 'type'], dtype='str')

In [9]:
# for the kaggle dataset: filter for benign and phishing

In [10]:
prim_df["type"].value_counts(normalize=True)

type
benign      0.684653
phishing    0.315347
Name: proportion, dtype: float64

In [11]:
import re
import pandas as pd
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestClassifier


# --- Feature extraction functions ---
def get_url_string(df):
    return df["url"]


def safe_urlparse(url):
    """Safely parse URL, returning defaults on error"""
    try:
        return urlparse(url)
    except ValueError:
        # Return a dummy ParseResult on error
        return urlparse("http://default.com")


def extract_url_features(df):
    urls = df["url"]
    features = pd.DataFrame()

    features["url_length"] = urls.str.len()
    features["num_dots"] = urls.str.count(r"\.")
    features["num_hyphens"] = urls.str.count(r"-")
    features["num_slashes"] = urls.str.count(r"/")
    features["num_digits"] = urls.str.count(r"\d")
    features["num_special_chars"] = urls.str.count(r"[@_!#$%^&*()<>?|}{~:]")
    features["has_https"] = urls.str.startswith("https").astype(int)
    features["has_ip"] = urls.str.contains(
        r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}"
    ).astype(int)
    features["has_at_symbol"] = urls.str.contains("@").astype(int)
    features["subdomain_count"] = urls.apply(
        lambda u: len(safe_urlparse(u).netloc.split(".")) - 2
    )
    features["path_length"] = urls.apply(lambda u: len(safe_urlparse(u).path))
    features["query_length"] = urls.apply(lambda u: len(safe_urlparse(u).query))

    return features


# --- Data ---
X = prim_df[["url"]]
y = prim_df["type"]

# --- Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# --- Pipeline ---
pipeline = Pipeline(
    [
        (
            "features",
            FeatureUnion(
                [
                    (
                        "tfidf",
                        Pipeline(
                            [
                                ("get_text", FunctionTransformer(get_url_string)),
                                (
                                    "vec",
                                    TfidfVectorizer(
                                        analyzer="char_wb",
                                        ngram_range=(3, 5),
                                        max_features=5000,
                                    ),
                                ),
                            ]
                        ),
                    ),
                    (
                        "handcrafted",
                        Pipeline(
                            [("get_feats", FunctionTransformer(extract_url_features))]
                        ),
                    ),
                ]
            ),
        ),
        (
            "clf",
            RandomForestClassifier(
                class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1
            ),
        ),
    ]
)

pipeline.fit(X_train, y_train)

# --- Evaluate ---
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

In [12]:
y_pred

array(['benign', 'benign', 'benign', ..., 'benign', 'benign', 'benign'],
      shape=(100997,), dtype=object)

In [13]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
import matplotlib.pyplot as plt
import seaborn as sns

# --- Evaluation Metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\n" + "=" * 50)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

MODEL PERFORMANCE METRICS
Accuracy:  0.9965
Precision: 0.9965
Recall:    0.9965
F1-Score:  0.9965

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

      benign       1.00      1.00      1.00     69148
    phishing       1.00      0.99      0.99     31849

    accuracy                           1.00    100997
   macro avg       1.00      1.00      1.00    100997
weighted avg       1.00      1.00      1.00    100997


Confusion Matrix:
[[69033   115]
 [  239 31610]]


In [14]:
# --- False Negatives Analysis ---
# False Negatives: actual phishing but predicted as benign
false_negatives = ((y_test == "phishing") & (y_pred == "benign")).sum()
true_positives = ((y_test == "phishing") & (y_pred == "phishing")).sum()
total_phishing = (y_test == "phishing").sum()

print("=" * 50)
print("FALSE NEGATIVES ANALYSIS")
print("=" * 50)
print(f"Total phishing URLs in test set: {total_phishing}")
print(f"True Positives (correctly caught): {true_positives}")
print(f"False Negatives (missed phishing): {false_negatives}")
print(f"Percentage missed: {(false_negatives / total_phishing * 100):.2f}%")
print("=" * 50)

FALSE NEGATIVES ANALYSIS
Total phishing URLs in test set: 31849
True Positives (correctly caught): 31610
False Negatives (missed phishing): 239
Percentage missed: 0.75%


In [15]:
kaggle_df["type"].value_counts()

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

In [16]:
# filter for the bengin and phising dataset from the kaggle
second_df = kaggle_df[kaggle_df["type"].isin(["benign", "phishing"])]

In [17]:
# Check if second dataset has similar structure
print("Second Dataset Info:")
print(f"Shape: {second_df.shape}")
print(f"Columns: {second_df.columns.tolist()}")
print()

# Make predictions on second dataset
X_second = second_df[["url"]]
y_second_pred = pipeline.predict(X_second)
y_second_proba = pipeline.predict_proba(X_second)[:, 1]

# If labels exist in second dataset, evaluate performance
if "type" in second_df.columns or "label" in second_df.columns:
    # Determine label column name
    label_col = "type" if "type" in second_df.columns else "label"
    y_second = second_df[label_col].str.lower()

    # Evaluate
    accuracy_second = accuracy_score(y_second, y_second_pred)
    precision_second = precision_score(
        y_second, y_second_pred, average="weighted", zero_division=0
    )
    recall_second = recall_score(
        y_second, y_second_pred, average="weighted", zero_division=0
    )
    f1_second = f1_score(y_second, y_second_pred, average="weighted", zero_division=0)

    print("=" * 50)
    print("SECOND DATASET - MODEL PERFORMANCE")
    print("=" * 50)
    print(f"Accuracy:  {accuracy_second:.4f}")
    print(f"Precision: {precision_second:.4f}")
    print(f"Recall:    {recall_second:.4f}")
    print(f"F1-Score:  {f1_second:.4f}")
    print("\n" + "=" * 50)
    print("CLASSIFICATION REPORT")
    print("=" * 50)
    print(classification_report(y_second, y_second_pred))
else:
    print("=" * 50)
    print("SECOND DATASET - PREDICTIONS ONLY")
    print("=" * 50)
    print(f"Total URLs predicted: {len(y_second_pred)}")
    print(f"Predicted as phishing: {(y_second_pred == 'phishing').sum()}")
    print(f"Predicted as benign: {(y_second_pred == 'benign').sum()}")
    print("\nPreview of predictions:")
    print(
        pd.DataFrame(
            {
                "url": X_second["url"].head(10),
                "prediction": y_second_pred[:10],
                "confidence": y_second_proba[:10],
            }
        )
    )

Second Dataset Info:
Shape: (522214, 2)
Columns: ['url', 'type']

SECOND DATASET - MODEL PERFORMANCE
Accuracy:  0.2028
Precision: 0.7901
Recall:    0.2028
F1-Score:  0.1034

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      benign       0.92      0.03      0.06    428103
    phishing       0.18      0.99      0.31     94111

    accuracy                           0.20    522214
   macro avg       0.55      0.51      0.18    522214
weighted avg       0.79      0.20      0.10    522214



In [18]:
# ============================================================
# LABEL ALIGNMENT CHECKS
# ============================================================

# 1. Check what classes the pipeline knows
print("Pipeline classes:", pipeline.classes_)

# 2. Check raw label values in both datasets
print("Train labels:  ", sorted(y_train.unique()))
print("Second labels: ", sorted(second_df[label_col].unique()))

# 3. Normalize labels to lowercase (you're already doing this for y_second,
#    but make sure training labels were also lowercase when fit)
print("Sample train labels:", y_train.head(5).tolist())
print("Sample second labels:", second_df[label_col].head(5).tolist())

# 4. Check predicted distribution vs actual
print("\nPredicted distribution:")
print(pd.Series(y_second_pred).value_counts(normalize=True))
print("\nActual distribution:")
print(second_df[label_col].str.lower().value_counts(normalize=True))

# 5. Spot check — look at high-confidence wrong predictions
results_df = pd.DataFrame(
    {
        "url": X_second["url"],
        "actual": y_second,
        "predicted": y_second_pred,
        "confidence": y_second_proba,
    }
)

wrong = results_df[results_df["actual"] != results_df["predicted"]]
print("\nHigh-confidence mistakes:")
print(wrong.sort_values("confidence", ascending=False).head(10))

Pipeline classes: ['benign' 'phishing']
Train labels:   ['benign', 'phishing']
Second labels:  ['benign', 'phishing']
Sample train labels: ['phishing', 'benign', 'benign', 'benign', 'phishing']
Sample second labels: ['phishing', 'benign', 'benign', 'benign', 'benign']

Predicted distribution:
phishing    0.973337
benign      0.026663
Name: proportion, dtype: float64

Actual distribution:
type
benign      0.819785
phishing    0.180215
Name: proportion, dtype: float64

High-confidence mistakes:
                                                      url  actual predicted  \
445425  http://cheezburger.com/8491717632/video-games-...  benign  phishing   
273659  http://ecnavi.jp/redirect/?url=http://ad-4091....  benign  phishing   
68418   http://metro.co.uk/2015/04/19/thats-awkward-lo...  benign  phishing   
440780  http://tinnhanh360.net/clip-cam-dong-chong-tre...  benign  phishing   
440784  http://io9.com/5867059/val-kilmers-meth-loving...  benign  phishing   
273720  http://katproxy.com/

In [19]:
import json, os

os.makedirs("../output", exist_ok=True)

# Phishing-class recall and FN rate from confusion matrix
fn = cm[1][0]
tp = cm[1][1]
fn_rate = fn / (fn + tp)
phish_recall = tp / (tp + fn)

# ROC-AUC on primary test set
primary_roc_auc = float(roc_auc_score((y_test == "phishing").astype(int), y_proba))

results = {
    "primary": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "phish_recall": float(phish_recall),
        "f1": float(f1),
        "roc_auc": primary_roc_auc,
        "confusion_matrix": cm.tolist(),
        "fn_rate": float(fn_rate),
    },
    "kaggle": {
        "accuracy": float(accuracy_second),
        "precision": float(precision_second),
        "recall": float(recall_second),
        "f1": float(f1_second),
    },
}

with open("../output/rf_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → output/rf_results.json")

Saved → output/rf_results.json


In [20]:
# --- Save RF pipeline ---
import joblib, os

os.makedirs("../output/models", exist_ok=True)
joblib.dump(pipeline, "../output/models/rf_pipeline.joblib")
print("Saved RF pipeline → output/models/rf_pipeline.joblib")

Saved RF pipeline → output/models/rf_pipeline.joblib
